In [1]:
import pandas as pd
import psycopg2

In [2]:
caminho_desafio_dw = 'C:\\Users\\PC\\Documents\\Desafio_dw_eletronicos\\dados\\desafio2_estoque.csv'
df_desafio_dw_estoque = pd.read_csv(caminho_desafio_dw, sep=',')
df_desafio_dw_estoque.head(1)

,produto_id,produto,categoria,marca,fornecedor,estoque_inicial,entradas_periodo,estoque_atual,estoque_minimo,custo_unitario,centro_distribuicao
0,101,Notebook Pro 14,Notebook,TechMax,Fornecedor TechMax,507,78,503,67,3196.94,CD Sul


In [3]:
#Verificar informações da tabela
df_desafio_dw_estoque.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   produto_id           12 non-null     int64  
 1   produto              12 non-null     str    
 2   categoria            12 non-null     str    
 3   marca                12 non-null     str    
 4   fornecedor           12 non-null     str    
 5   estoque_inicial      12 non-null     int64  
 6   entradas_periodo     12 non-null     int64  
 7   estoque_atual        12 non-null     int64  
 8   estoque_minimo       12 non-null     int64  
 9   custo_unitario       12 non-null     float64
 10  centro_distribuicao  12 non-null     str    
dtypes: float64(1), int64(5), str(5)
memory usage: 1.2 KB


In [4]:
dbname = 'dw_eletronicos'
user = 'postgres'
password = '12345'
host = 'localhost'
port = '5432'

conexao = psycopg2.connect(dbname=dbname, user=user, password=password, host=host, port=port)
cursor = conexao.cursor()

## CRIANDO TABELA raw_vendas NO BANCO DE DADOS
cursor.execute(""" 
               
                CREATE TABLE IF NOT EXISTS raw.raw_estoque
                    (
                        produto_id integer,
                        produto varchar(100),
                        categoria varchar(50),
                        marca varchar(50),
                        fornecedor varchar(100),
                        estoque_inicial integer,
                        entradas_periodo integer,
                        estoque_atual integer,
                        estoque_minimo integer,
                        custo_unitario decimal(10,2),
                        centro_distribuicao varchar(50)
                        
                    );

               """)
conexao.commit()
cursor.close()
conexao.close()

In [5]:
dbname = 'dw_eletronicos'
user = 'postgres'
password = '12345'
host = 'localhost'
port = '5432'

conexao = psycopg2.connect(dbname=dbname,user=user,password=password,host=host,port=port)
cursor = conexao.cursor()

cursor.execute('delete from raw.raw_estoque')

for i, df_desafio_dw_estoque_raw in df_desafio_dw_estoque.iterrows():
    cursor.execute(""" insert into raw.raw_estoque(
                   produto_id, 
                   produto, 
                   categoria, 
                   marca, 
                   fornecedor,
                   estoque_inicial, 
                   entradas_periodo, 
                   estoque_atual,
                    estoque_minimo, 
                   custo_unitario, 
                   centro_distribuicao
                   ) values (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                   """ ,(
                       df_desafio_dw_estoque_raw['produto_id'],
                       df_desafio_dw_estoque_raw['produto'],
                       df_desafio_dw_estoque_raw['categoria'],
                       df_desafio_dw_estoque_raw['marca'],
                       df_desafio_dw_estoque_raw['fornecedor'],
                       df_desafio_dw_estoque_raw['estoque_inicial'],
                       df_desafio_dw_estoque_raw['entradas_periodo'],
                       df_desafio_dw_estoque_raw['estoque_atual'],
                       df_desafio_dw_estoque_raw['estoque_minimo'],
                       df_desafio_dw_estoque_raw['custo_unitario'],
                       df_desafio_dw_estoque_raw['centro_distribuicao']
                   ))
conexao.commit()
cursor.close()
conexao.close()

In [6]:
dbname = 'dw_eletronicos'
user = 'postgres'
password = '12345'
host = 'localhost'
port = '5432'

conexao = psycopg2.connect(dbname=dbname, user=user, password=password, host=host, port=port)
cursor = conexao.cursor()

cursor.execute("""

                CREATE TABLE IF NOT EXISTS staging.stg_estoque as 
                    select distinct 
                        produto_id,
                        produto,
                        categoria,
                        lower(marca) as marca_produto,
                        trim(fornecedor) as fornecedor,
                        estoque_inicial,
                        entradas_periodo,
                        estoque_atual,
                        estoque_minimo,
                        custo_unitario :: numeric(10,2),
                        Upper(centro_distribuicao) as centro_distribuicao
                    From raw.raw_estoque
                    Where produto is not null;

               
               """)
conexao.commit()
cursor.close()
conexao.close()